# Subm3 - g8 MEI B

Submissão `B` para a Subm3 com o modelo `DNN numpy`.


In [ ]:
import os
import sys
import pickle

sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd

from src.data_processing import clean_text
from src.models_numpy.dnn import CategoricalCrossEntropy, Dataset, NeuralNetwork, accuracy


In [ ]:
ROOT = os.path.abspath('..')
CLASSES = ['Anthropic', 'Google', 'Human', 'Meta', 'OpenAI']

SUBM3_INPUT_PATH = os.path.join(ROOT, 'Subm3', 'validation_data', 'subm3.csv')
SUBM3_OUTPUT_PATH = os.path.join(ROOT, 'Subm3', 'subm3-g8-MEI-B.csv')
SAVED_MODELS_DIR = os.path.join(ROOT, 'saved_models')


## Load Subm3


In [ ]:
df_subm3 = pd.read_csv(SUBM3_INPUT_PATH, sep=';')
print(df_subm3.head())
print(f'Total rows: {len(df_subm3)}')


## Load DNN


In [ ]:
dnn_model = NeuralNetwork(loss=CategoricalCrossEntropy, metric=accuracy, verbose=False)
dnn_model.load(os.path.join(SAVED_MODELS_DIR, 'dnn_final_model.npz'))

with open(os.path.join(SAVED_MODELS_DIR, 'dnn_final_vectorizer.pkl'), 'rb') as f:
    dnn_vectorizer = pickle.load(f)


def predict_dnn(model, vectorizer, texts_raw):
    texts_clean = [clean_text(text) for text in texts_raw]
    X = vectorizer.transform(texts_clean, texts_raw)
    return model.predict(Dataset(X, np.zeros((len(texts_raw), len(CLASSES)))))


## Predict and Save CSV


In [ ]:
dnn_probs = predict_dnn(dnn_model, dnn_vectorizer, df_subm3['Text'].tolist())
pred_ids = np.argmax(dnn_probs, axis=1)
pred_labels = [CLASSES[idx] for idx in pred_ids]

submission = pd.DataFrame({
    'ID': df_subm3['ID'],
    'Label': pred_labels,
})
submission.to_csv(SUBM3_OUTPUT_PATH, sep=';', index=False)

print(submission.head())
print(f'Saved to {SUBM3_OUTPUT_PATH}')
